# WEEK 7｜Pandas Series、DataFrame 与缺失数据

本周两份讲义介绍 Pandas 的 Series 与 DataFrame：如何创建和选取数据、索引如何参与运算、以及缺失值的识别、删除和填补。

## 1. Pandas 数据结构

Series 是带显式索引的一维数据；DataFrame 是带行索引和列名的二维表，可以看作多个共享行索引的 Series 列。NumPy 数组主要通过位置索引；Pandas 还可使用有意义的标签索引。

In [ ]:
import numpy as np
import pandas as pd

area = pd.Series(
    {"California": 423967, "Texas": 695662, "New York": 141297},
    name="area",
)
print(area)
print(area.index)
print(area.values)

## 2. Series：值与索引

Series 可以由列表、NumPy 数组、字典或单个标量创建。若未给 index，Pandas 使用整数索引；若给了字典，默认用键作为索引。为标量指定多个 index 时，该值会填入每个索引位置。Series 可按标签读写，也可用布尔条件或一组标签选择。

In [ ]:
scores = pd.Series([78, 91, 85], index=["Amy", "Bo", "Cy"])
from_dict = pd.Series({"Amy": 78, "Bo": 91, "Cy": 85})
filled = pd.Series(0, index=["a", "b", "c"])

print(scores["Bo"])
print(scores[scores > 80])
print(scores[["Amy", "Cy"]])
scores["Dee"] = 88
print(scores)

### 标签索引与位置索引

Series 同时有显式标签与隐式整数位置。标签索引用 `loc`，位置索引用 `iloc`。如果索引本身是整数，直接写 `data[1]` 会按标签查找；切片的行为则可能按位置解释。明确使用 loc/iloc 可避免混淆。标签切片包含终点标签，位置切片遵循 Python 的起点包含、终点不包含。

In [ ]:
data = pd.Series([10, 20, 30], index=[1, 3, 5])
print(data.loc[1], data.loc[1:3])
print(data.iloc[0], data.iloc[0:2])

Series 也可像字典一样查看 `keys()`、`items()`，并检查某个标签是否存在；同时它支持数组式切片和运算。讲义用 Series 表达“索引到值”的映射，并指出 Series 的索引和值有相应数据类型。

## 3. DataFrame：行列标签与构造

DataFrame 可由 Series 字典、字典列表、二维 NumPy 数组等构造。由不同索引的 Series 组成时，Pandas 按标签对齐；缺少对应标签的位置填为缺失值 NaN。可以指定行索引和列名，也可用 `head()` 查看前几行。

In [ ]:
area = pd.Series({"California": 423967, "Texas": 695662})
population = pd.Series({"Texas": 26448193, "New York": 19651127})
states = pd.DataFrame({"area": area, "population": population})
print(states)
print(states.head(2))

选择列最直接的方式是 `df['column']`，结果为 Series；若列名符合属性命名条件，也可写 `df.column`，但讲义建议优先用方括号。行按标签用 `loc`；`iloc` 按行列整数位置。

In [ ]:
data = pd.DataFrame(
    {"area": [4, 7, 2], "population": [10, 20, 15]},
    index=["A", "B", "C"],
)
print(data["area"])
print(data.loc["A":"B", ["area", "population"]])
print(data.iloc[:2, :1])

DataFrame 可添加计算列、修改单元格或删除行列。布尔条件用于筛选行；也可在 loc 中同时指定筛选条件和要显示的列。方括号中单个列名用于取列，切片和布尔条件常用于选择行。

In [ ]:
data["density"] = data["population"] / data["area"]
print(data.loc[data.density > 2, ["population", "density"]])
without_c = data.drop("C")
del data["area"]
print(data)

## 4. 索引对齐与运算

Pandas 运算按索引标签对齐，而不是只按当前显示位置配对。两个 Series 相加时，结果索引是两边索引的并集；只在一侧出现的标签会产生 NaN。需要用指定值代替缺失配对时，可用 `add(other, fill_value=...)`。

In [ ]:
x = pd.Series([2, 4, 6], index=[0, 1, 2])
y = pd.Series([1, 3, 5], index=[1, 2, 3])
print(x + y)
print(x.add(y, fill_value=0))

DataFrame 同样按行列标签对齐，列顺序不同也会正确按列名匹配。DataFrame 与 Series 运算时，默认按列标签进行广播；指定 `axis=0` 可沿行方向匹配 Series。常用逐元素方法包括 add、sub、mul、div、floordiv、mod 和 pow。

In [ ]:
X = pd.DataFrame({"A": [1, 2], "B": [3, 4]})
Y = pd.DataFrame({"B": [10, 20], "A": [30, 40]})
print(X + Y)

row = X.iloc[0]
print(X - row)                    # 按列对齐
print(X.subtract(X["A"], axis=0)) # 按行对齐)

## 5. None、NaN 与缺失值

None 是 Python 的空值对象；NaN 是浮点数中表示“非数值”的特殊值。NaN 不等于自身，所以不宜用 `==` 判断 NaN；用 `isna()/isnull()` 检查，`notna()/notnull()` 得到相反掩码。含 NaN 的普通算术结果通常仍为 NaN，聚合可使用忽略 NaN 的专门函数或 Pandas 聚合。

In [ ]:
value = np.nan
print(np.isnan(value))
print(value == value)  # False

s = pd.Series([1, np.nan, 2, None])
print(s.isna())
print(s.notna())

缺失值出现时，Pandas 会根据数据类型容纳它：整数数据出现 NaN 时可能转换为浮点类型；布尔数据遇到缺失值时可能转换为 object。讲义强调，缺失值对列类型的影响取决于该类型是否有可用的缺失值表示。

## 6. 检查、删除与填补缺失值

`dropna()` 返回移除缺失数据后的结果；默认会删除含缺失值的行。用 `axis=1` 删除含缺失值的列，`how='all'` 只删除全为空的行或列，`thresh` 指定至少需要多少个非空值。原对象由返回的新结果替代保存，便于保留处理前数据。

In [ ]:
df = pd.DataFrame([[1, np.nan, 2], [2, 3, 5], [np.nan, 4, 6]])
print(df.dropna())
print(df.dropna(axis=1))
print(df.dropna(axis=1, how="all"))
print(df.dropna(thresh=2))

`fillna(value)` 用指定值填补；`ffill()` 用前一个有效值向后填充，`bfill()` 用后一个有效值向前填充。DataFrame 可指定 axis 决定沿行或列方向填充。

In [ ]:
series = pd.Series([1, np.nan, 2, None, 3], index=list("abcde"))
print(series.fillna(0))
print(series.ffill())
print(series.bfill())

source 的 Series 练习还要求把 Series 中的值转换为 Python list；读取时要注意结果按 Series 项目顺序排列。

## 本周回顾

Series 是带索引的一维结构，DataFrame 是带行列标签的二维结构。选择时区分标签 loc 与位置 iloc；运算按标签自动对齐，不匹配处会产生 NaN。缺失值用 isna 检查，再按需要 dropna 删除或 fillna/ffill/bfill 填补。